![](TPCH/Screenshot 2026-10-22 153015.png)

## Scheduling and Monitoring the Pipeline in Production

### Scheduling

To run this pipeline in production, I would convert the notebook into a **Databricks Job** with a **scheduled trigger**, for example, running every night at 2 AM so that the daily batch of data is ingested and processed automatically during off-peak hours.

Key scheduling considerations:

- **Trigger type**: *Schedule* — set to run nightly at a fixed time when source data is expected to be available and compute costs are lower.
- **Concurrency**: Set *Maximum concurrent runs* to 1 to prevent overlapping runs corrupting the same tables.
- **Retries**: Enable automatic retries (e.g., 2 retries with a short backoff interval) so transient failures (network blips, source unavailability) self-heal without paging anyone.
- **Alerting on late data**: If the source might be late, add a small pre-task or use file arrival sensors rather than assuming the data is ready at the scheduled time.

### Monitoring

Once scheduled, I would monitor the pipeline using:

- **Job run history**: The Jobs UI shows each run's status, duration, and logs — the first place to check when a run fails or slows down.
- **Email / Slack notifications**: Configure job notifications on *run failure* and (optionally) *run success* so the on-call team is alerted immediately instead of discovering the problem downstream.
- **Job-level alerts**: Set up alerts for runs that exceed a duration threshold, since a slow run can be an early warning of data volume growth or skewed processing.
- **System tables**: Query `system.lakeflow` (job/pipeline run history and performance) and `system.billing.usage` to track run frequency, failure rates, and cost per run — useful for spotting regressions and justifying compute sizing.
- **Data quality checks**: Include validation steps in the job (row counts, null checks, freshness checks) that fail the run loudly if the data looks wrong, so bad data never silently propagates downstream.
- **Backfills**: For a missed or failed run, trigger a manual re-run / backfill of the affected date range rather than disabling the schedule.

In summary: schedule the job nightly, retry transient failures automatically, alert on failure and latency, and monitor run history and system tables to keep the pipeline healthy and cost-effective. 

#### Three key findings, each supported by a specific query result and metric value.

Three Key Business Findings from TPC-H Sales Data
Based on analysis of the TPC-H sales dataset covering 1992-1998, here are three key findings with specific metrics:
1. Stable Annual Revenue Performance at $165B
The business generated remarkably consistent annual net revenue of approximately $165.5 billion across full years 1992-1997, with minimal year-over-year variation (less than 1%). The average annual revenue was $165.5B, with individual years ranging from $165.0B (1995) to $166.0B (1996). This exceptional stability suggests a mature, predictable business model with consistent demand patterns.


In [0]:
SELECT
  YEAR(order_date) AS order_year,
  SUM(extended_price * (1 - discount_rate)) AS net_revenue,
  COUNT(DISTINCT order_key) AS order_count,
  COUNT(DISTINCT customer_key) AS active_customers,
  SUM(extended_price * discount_rate) AS discount_amount,
  SUM(extended_price * (1 - discount_rate)) / COUNT(DISTINCT order_key) AS avg_net_order_value
FROM
  `tpch_dev`.`gold`.`fact_sales`
GROUP BY
  YEAR(order_date)
ORDER BY
  order_year

2. Balanced Market Segment Distribution with Furniture Leading
All five market segments contribute nearly equally to total revenue, demonstrating excellent diversification. FURNITURE leads with $199.6B in net revenue (1992-1997), followed closely by BUILDING at $199.3B, MACHINERY at $198.4B, AUTOMOBILE at $198.2B, and HOUSEHOLD at $197.5B. The difference between the highest and lowest segments is only $2.1B (approximately 1%), indicating no single segment dominates or underperforms.

In [0]:
SELECT
  customer_market_segment,
  SUM(extended_price * (1 - discount_rate)) AS net_revenue,
  COUNT(DISTINCT order_key) AS order_count,
  COUNT(DISTINCT customer_key) AS active_customers,
  SUM(extended_price * (1 - discount_rate)) / COUNT(DISTINCT order_key) AS avg_net_order_value
FROM
  `tpch_dev`.`gold`.`fact_sales`
WHERE
  YEAR(order_date) BETWEEN 1992 AND 1997
GROUP BY
  customer_market_segment
ORDER BY
  net_revenue DESC

3. Europe Leads Geographic Revenue with Minimal Regional Variation
EUROPE generates the highest regional revenue at $199.5B (1992-1997), but the geographic distribution is remarkably balanced. MIDDLE EAST follows at $199.3B, ASIA at $198.9B, AMERICA at $197.7B, and AFRICA at $197.4B. The spread between top and bottom regions is only $2.0B (approximately 1%), demonstrating successful global market penetration with no region significantly lagging.3

In [0]:
SELECT
  g.region_name,
  SUM(f.extended_price * (1 - f.discount_rate)) AS net_revenue,
  COUNT(DISTINCT f.order_key) AS order_count,
  COUNT(DISTINCT f.customer_key) AS active_customers,
  SUM(f.extended_price * (1 - f.discount_rate)) / COUNT(DISTINCT f.order_key) AS avg_net_order_value
FROM
  `tpch_dev`.`gold`.`fact_sales` f
    JOIN `tpch_dev`.`gold`.`dim_geography` g
      ON f.customer_nation_key = g.geography_key
WHERE
  YEAR(f.order_date) BETWEEN 1992 AND 1997
GROUP BY
  g.region_name
ORDER BY net_revenue

#### Two opportunities for growth or operational improvement, with data evidence.


Two Strategic Opportunities for Growth and Improvement
Based on analysis of the TPC-H sales data (1992-1997), here are two concrete opportunities with quantifiable impact:
1. Optimize Discount Strategy: $10.4B Annual Revenue Opportunity

The business currently applies a uniform 5% discount rate across all transactions, totaling $52.3B in annual discounts. This one-size-fits-all approach represents a significant margin optimization opportunity.

The Opportunity: Reducing the average discount rate by just 1 percentage point (from 5% to 4%) would generate $10.4B in additional annual net revenue — a 1.05% revenue increase with no additional customer acquisition costs.
Recommended Actions:
- Segment-based pricing: The discount rate is virtually identical across all five market segments (4.997% to 5.001%). Implement differentiated discount strategies based on segment price sensitivity and competitive dynamics.
Customer tier pricing: High-frequency customers (11+ orders) already demonstrate loyalty and may not require the same discount level as acquisition-focused offers for new customers.
- Volume-based discounting: Replace blanket discounts with targeted incentives tied to order size or annual spend thresholds.

This represents low-hanging fruit — the business can capture significant margin improvement through smarter pricing without sacrificing volume, as the current uniform approach suggests discounts are not strategically deployed.

2. Increase Customer Purchase Frequency: Massive Revenue Upside
Customer engagement analysis reveals a stark divide: 64% of customers (319,768) place 11+ orders and generate 80% of total revenue ($793.8B), while low-frequency customers contribute minimally.3
Customer Revenue by Purchase Frequency
Source:
- Customer Purchase Frequency Distribution
-The Opportunity: Customers who order 11+ times generate $2.48M in average lifetime revenue compared to just $133K for single-order customers — an 18.7x difference. Moving even a small portion of low-frequency customers up the engagement ladder would drive substantial growth.3
- Current State:
Average orders per customer across all years: 2.63 orders over the 6-year period
45,477 customers (9%) have placed 6 or fewer orders — these are at-risk or underengaged4
Recommended Actions:
Retention programs: Implement targeted campaigns for customers after their first 1-3 orders to drive repeat purchases (the critical transition zone).
Regional focus: Europe has the highest revenue per customer at $1.99M, while Africa trails at $1.98M. While the gap is small (0.3%), replicating Europe's best practices in customer engagement could lift performance in other regions.5
Revenue per Customer by Region
Source:
Regional Customer Engagement Metrics
Lifecycle marketing: The data shows customers average only 2.63 orders over 6 years. Increasing this to 3 orders per customer would add approximately 190K orders annually — worth ~$27.6B in additional revenue at current order values.
Impact Estimate: If just 10% of the 45,477 low-frequency customers (4,548 customers) increased their order frequency from an average of 4 orders to 8 orders over the period, this would generate approximately $1.4B in incremental revenue (4,548 customers × 4 additional orders × $145K average order value).

#### two risk areas requiring further investigation with data evidence

Two Risk Areas Requiring Investigation
1. Revenue Recognition Risk: $465B in Unfulfilled Orders

46.8% of total revenue ($465B) remains in Open or Pending status, representing significant revenue recognition and cash flow exposure
- Open orders (O): 2.99M orders worth $430.8B (43.4% of revenue)
- Pending orders (P): 191K orders worth $34.0B (3.4% of revenue)
- These orders average 30+ years old (placed 1992-1997, still not fulfilled as of 2026)

**Investigation needed**: This represents either a massive revenue recognition issue (revenue booked but not delivered), potential bad debt, or a data quality problem. The business needs to immediately investigate whether these represent legitimate backorders, disputed orders, or accounting errors that overstate reported revenue.

2. Extreme Customer Fragmentation: No Concentration Protection

The customer base shows unusual fragmentation with zero meaningful concentration risk — the top 10,000 customers (2% of the base) generate only 4.4% of revenue.
- Top 100 customers: 0.06% of revenue
- Top 1,000 customers: 0.51% of revenue
- Remaining 490K customers: 95.6% of revenue

**Investigation needed:** This distribution is highly unusual for B2B operations and suggests either: (1) a true consumer-scale business with millions of small transactions, (2) data quality issues where customer identifiers are fragmented/duplicated, or (3) the business lacks any strategic accounts or key customer relationships. The company should verify customer master data integrity and assess whether this extreme fragmentation limits pricing power and increases operational complexity.



1 source


#### A recommended management action for each finding, opportunity, and risk

Key Findings

1. Stable $165B Annual Revenue
- Action: Shift focus from top-line growth to margin expansion and operational efficiency. With flat revenue, prioritize initiatives that improve profitability per dollar of sales.
2. Balanced Market Segments (5 segments within 1%)
- Action: Maintain current diversification but establish segment-specific growth targets. Avoid over-indexing resources on any single segment; instead, replicate best practices across all five.
3. Balanced Regional Distribution (5 regions within 1%)
- Action: Benchmark Europe's customer engagement practices (highest revenue per customer at $1.99M) and deploy successful tactics to other regions to lift overall performance.


Growth Opportunities
1. Discount Optimization ($10.4B opportunity)
- Action: Immediately launch a 90-day pilot in one market segment testing tiered discount rates (3-6% range) based on customer lifetime value and order size. Measure price elasticity before rolling out enterprise-wide.
2. Increase Purchase Frequency (18.7x revenue difference)
- Action: Deploy automated retention campaigns targeting customers after orders 1-3 (the critical drop-off zone). Offer incentives for second purchases within 90 days and establish quarterly business reviews for customers approaching 10+ order status.


Risk Areas
1. $465B Unfulfilled Orders (46.8% of revenue)
- Action: Convene an emergency cross-functional task force (Finance, Operations, IT) within 48 hours to audit order status data integrity. Reclassify or write off aged open/pending orders and implement monthly order aging reports with executive oversight.
2. Extreme Customer Fragmentation (Top 1K = 0.5% of revenue)
- Action: Commission an immediate customer master data quality audit to identify duplicate records and consolidate customer identities. Simultaneously, develop a strategic accounts program to cultivate larger relationships and reduce operational complexity.